<span style="font-size:2em; font-weight:bold;">Objective 2: Evaluation of Ground Motion Spatial Correlation in New Zealand Through Empirical Approaches - Model Selection</span>
***

This code was written as part of the electronic supplement which deals with feature and model selection of the extended non-stationary spatial correlation models.

# 1.0 Import Modules and Data

## 1.1 Import Modules

In [ ]:
# Repository paths (works when Jupyter starts at the repository root or in notebooks/).
from pathlib import Path
import sys

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (parent for parent in (_current, *_current.parents)
     if (parent / "src" / "cbb21_cossim.py").is_file()
     and (parent / "data").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Repository root not found. Start Jupyter in the repository or notebooks/ directory.")

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = REPO_ROOT / "data"
RESIDUALS_DIR = DATA_DIR / "residuals"
SCM_DIR = DATA_DIR / "nzwide_scm"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import math
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)
import matplotlib.ticker as mtick
from matplotlib.ticker import AutoMinorLocator, LogLocator, NullFormatter, FuncFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.path import Path as MplPath
from pyproj import Geod
from scipy.stats import bootstrap, gaussian_kde
from scipy.special import logsumexp

# scikitlearn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import CCA
import seaborn as sns

# arviz
import arviz as az

## 1.2 Import Data

### 1.2.1 Ground-motion database

In [ ]:
## Load NZ GMDB Flatfile
gm_im_table_50 = pd.read_csv(RESIDUALS_DIR / "ground_motion_im_table_rotd50_flat.csv", low_memory=False, dtype={"sta": "string"})

## Remove non-IM information to optimize storage
gm_im_table_50.drop(gm_im_table_50.iloc[:,182:571],inplace=True,axis = 1)

## Remove Duplicates BUT with HN channels as priority
gm_im_table_50.sort_values('chan',ascending=False,inplace=True)
gm_im_table_50.drop_duplicates(subset=['evid','sta'],keep='first',inplace=True)
gm_im_table_50.sort_values('datetime',ascending=True,inplace=True)

## Remove possible error values
gm_im_table_50 = gm_im_table_50[gm_im_table_50["pSA_0.01"]>0]

## append additional station parameters
stations = pd.read_csv(RESIDUALS_DIR / "site_table_TRI_multiscale_group_TWI.csv", dtype={"sta": "string"})
stations.fillna(0,inplace=True)
stations_filt = pd.DataFrame()
stations_filt['sta'] = stations['sta']
stations_filt['TRI'] = stations['TRI']
stations_filt['TWI'] = stations['TWI']
stations_filt['geoage'] = stations['geoMA_min']
stations_filt['k0'] = stations['k0_mean']
stations_filt['precip'] = stations['precip']
stations_filt['group'] = stations['cregion_group']
gm_im_table_50 = gm_im_table_50.merge(stations_filt,on='sta',how='left')

# apply maxmimum distance filter
gm_im_table_50 = gm_im_table_50[gm_im_table_50.r_jb<=300].reset_index(drop=True)

## Apply filter on the data: Tectonic Classification
gm_im_table_50_crustal = gm_im_table_50[(gm_im_table_50.tect_class=='Crustal')&(gm_im_table_50.mag>=3.5)]
gm_im_table_50_slab = gm_im_table_50[(gm_im_table_50.tect_class=='Slab')&(gm_im_table_50.mag>=5)]
gm_im_table_50_interface = gm_im_table_50[(gm_im_table_50.tect_class=='Interface')&(gm_im_table_50.mag>=5)]
gm_im_table_50 = pd.concat([gm_im_table_50_crustal, gm_im_table_50_slab, gm_im_table_50_interface],axis=0)
gm_im_table_50 = gm_im_table_50.reset_index(drop = True)

# get event ids for crustal
evid_crus = gm_im_table_50_crustal['evid'].unique().tolist()

### 1.2.1 Within-event residuals

In [ ]:
# residuals
im_dwbar_t6 = pd.read_csv(RESIDUALS_DIR / "NZGMDB_v4p3_dwbar_T6.csv", dtype={"sta": "string"})
selected_columns = ['sta', 'Z2.5', 'hriver_dist']
im_dwbar_t6 = im_dwbar_t6.merge(stations[selected_columns], on='sta', how='left')

### 1.2.2 Model Comparison, LPPDs and directory

In [ ]:
# specify periods and prefix
pSA_T  = [0.01, 0.03, 0.06, 0.1, 0.3, 0.6, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
pSAobs_T = ['PGA', 'pSA_0.03', 'pSA_0.06', 'pSA_0.1','pSA_0.3', 'pSA_0.6', 'pSA_1.0', 'pSA_2.0', 'pSA_3.0', 'pSA_4.0', 'pSA_5.0', 'pSA_6.0', 'pSA_7.0', 'pSA_8.0', 'pSA_9.0', 'pSA_10.0']
pSA_names = ['PGA','pSA_003','pSA_006','pSA_010','pSA_030','pSA_060','pSA_100','pSA_200','pSA_300','pSA_400','pSA_500','pSA_600','pSA_700','pSA_800','pSA_900','pSA_1000']

# directory
directory = SCM_DIR

# Pooled LPPDs
LPPDP_reldiff = pd.read_csv(directory/'NZGMDB_v4p3_dwbar_T6_LPPDP.csv')
LPPDP_reldiff_lnmodels = pd.read_csv(directory/'NZGMDB_v4p3_dwbar_T6_lnmodels_LPPDP.csv')
LPPDES = pd.read_csv(directory/'NZGMDB_v4p3_dwbar_T6_LPPDPES.csv')

# filename prefix
modelE_file_name = f"NZGMDB_v4p3_dwbar_T6"

# model comparisons - PSIS-LOO-CV (linear model)
model_compare = []
for period in pSAobs_T[:12]:
    model_copmare_temp = pd.read_csv(directory.joinpath('%s_LOOCompare_%s.csv'%(modelE_file_name,period)))
    model_copmare_temp= model_copmare_temp.rename(columns={"Unnamed: 0": "model"})
    model_compare.append(model_copmare_temp)

model_compare_ln = []
for period in pSAobs_T[:12]:
    model_copmare_temp = pd.read_csv(directory.joinpath('%s_lnmodels_LOOCompare_%s.csv'%(modelE_file_name,period)))
    model_copmare_temp= model_copmare_temp.rename(columns={"Unnamed: 0": "model"})
    model_compare_ln.append(model_copmare_temp)

# event-specific LPPDs (linear model)
cols = {}

for period in pSAobs_T[:12]:

    pooled_ME_LPPDE = pd.read_csv(directory.joinpath('%s_LPPDE_E_%s.csv' % (modelE_file_name, period)))
    pooled_MEA_LPPDE = pd.read_csv(directory.joinpath('%s_LPPDE_EA_%s.csv' % (modelE_file_name, period)))
    pooled_MEAS_LPPDE = pd.read_csv(directory.joinpath('%s_LPPDE_EAS_%s.csv' % (modelE_file_name, period)))
    pooled_MEASZ_LPPDE = pd.read_csv(directory.joinpath('%s_LPPDE_EASZ_%s.csv' % (modelE_file_name, period)))
    pooled_MEAST_LPPDE = pd.read_csv(directory.joinpath('%s_LPPDE_EAST_%s.csv' % (modelE_file_name, period)))
    pooled_MEASP_LPPDE = pd.read_csv(directory.joinpath('%s_LPPDE_EASP_%s.csv' % (modelE_file_name, period)))
    pooled_MEASTWI_LPPDE = pd.read_csv(directory.joinpath('%s_LPPDE_EASTWI_%s.csv' % (modelE_file_name, period)))
    pooled_MEATWIGEO_LPPDE = pd.read_csv(directory.joinpath('%s_LPPDE_EATWIGEO_%s.csv' % (modelE_file_name, period)))
    pooled_MEASZT_LPPDE = pd.read_csv(directory.joinpath('%s_LPPDE_EASZT_%s.csv' % (modelE_file_name, period)))

    evid = pooled_ME_LPPDE.columns.str.replace('z_', '', regex=False)

    cols['E_%s' % period] = pd.Series(-np.log(len(pooled_ME_LPPDE)) + logsumexp(pooled_ME_LPPDE.to_numpy(), axis=0),index=evid)
    cols['EA_%s' % period] = pd.Series(-np.log(len(pooled_MEA_LPPDE)) + logsumexp(pooled_MEA_LPPDE.to_numpy(), axis=0),index=evid)
    cols['EAS_%s' % period] = pd.Series(-np.log(len(pooled_MEAS_LPPDE)) + logsumexp(pooled_MEAS_LPPDE.to_numpy(), axis=0),index=evid)
    cols['EASZ_%s' % period] = pd.Series(-np.log(len(pooled_MEASZ_LPPDE)) + logsumexp(pooled_MEASZ_LPPDE.to_numpy(), axis=0),index=evid)
    cols['EAST_%s' % period] = pd.Series(-np.log(len(pooled_MEAST_LPPDE)) + logsumexp(pooled_MEAST_LPPDE.to_numpy(), axis=0), index=evid)
    cols['EASP_%s' % period] = pd.Series(-np.log(len(pooled_MEASP_LPPDE)) + logsumexp(pooled_MEASP_LPPDE.to_numpy(), axis=0),index=evid)
    cols['EASTWI_%s' % period] = pd.Series(-np.log(len(pooled_MEASTWI_LPPDE)) + logsumexp(pooled_MEASTWI_LPPDE.to_numpy(), axis=0),index=evid)
    cols['EATWIGEO_%s' % period] = pd.Series(-np.log(len(pooled_MEATWIGEO_LPPDE)) + logsumexp(pooled_MEATWIGEO_LPPDE.to_numpy(), axis=0),index=evid)
    cols['EASZT_%s' % period] = pd.Series(-np.log(len(pooled_MEASZT_LPPDE)) + logsumexp(pooled_MEASZT_LPPDE.to_numpy(), axis=0),index=evid)

PO_LPPDE = pd.DataFrame(cols)
PO_LPPDE.index.name = 'evid'
PO_LPPDE = PO_LPPDE.reset_index()

# ensure correct ordering
order = LPPDES['evid'].astype(str)
PO_LPPDE['evid'] = PO_LPPDE['evid'].astype(str)
PO_LPPDE = PO_LPPDE.set_index('evid').loc[order].reset_index()

# event-specific LPPDs (log model)
cols_ln = {}

for period in pSAobs_T[:12]:

    pooled_ME_LPPDE = pd.read_csv(directory.joinpath('%s_LPPDE_E_%s.csv' % (modelE_file_name, period)))
    pooled_MEA_LPPDE = pd.read_csv(directory.joinpath('%s_LPPDE_EA_%s.csv' % (modelE_file_name, period)))
    pooled_MEAS_LPPDE_ln = pd.read_csv(directory.joinpath('%s_LPPDE_EAlnS_%s.csv' % (modelE_file_name, period)))
    pooled_MEASZ_LPPDE_ln = pd.read_csv(directory.joinpath('%s_LPPDE_EAlnSZ_%s.csv' % (modelE_file_name, period)))
    pooled_MEAST_LPPDE_ln = pd.read_csv(directory.joinpath('%s_LPPDE_EAlnST_%s.csv' % (modelE_file_name, period)))
    pooled_MEASP_LPPDE_ln = pd.read_csv(directory.joinpath('%s_LPPDE_EAlnSP_%s.csv' % (modelE_file_name, period)))
    pooled_MEASTWI_LPPDE_ln = pd.read_csv(directory.joinpath('%s_LPPDE_EAlnSTWI_%s.csv' % (modelE_file_name, period)))
    pooled_MEATWIGEO_LPPDE_ln = pd.read_csv(directory.joinpath('%s_LPPDE_EAlnTWIGEO_%s.csv' % (modelE_file_name, period)))
    pooled_MEASZT_LPPDE_ln = pd.read_csv(directory.joinpath('%s_LPPDE_EAlnSZT_%s.csv' % (modelE_file_name, period)))

    evid = pooled_ME_LPPDE.columns.str.replace('z_', '', regex=False)

    cols_ln['E_%s' % period] = pd.Series(-np.log(len(pooled_ME_LPPDE)) + logsumexp(pooled_ME_LPPDE.to_numpy(), axis=0),index=evid)
    cols_ln['EA_%s' % period] = pd.Series(-np.log(len(pooled_MEA_LPPDE)) + logsumexp(pooled_MEA_LPPDE.to_numpy(), axis=0),index=evid)
    cols_ln['EAlnS_%s' % period] = pd.Series(-np.log(len(pooled_MEAS_LPPDE_ln)) + logsumexp(pooled_MEAS_LPPDE_ln.to_numpy(), axis=0),index=evid)
    cols_ln['EAlnSZ_%s' % period] = pd.Series(-np.log(len(pooled_MEASZ_LPPDE_ln)) + logsumexp(pooled_MEASZ_LPPDE_ln.to_numpy(), axis=0),index=evid)
    cols_ln['EAlnST_%s' % period] = pd.Series(-np.log(len(pooled_MEAST_LPPDE_ln)) + logsumexp(pooled_MEAST_LPPDE_ln.to_numpy(), axis=0),index=evid)
    cols_ln['EAlnSP_%s' % period] = pd.Series(-np.log(len(pooled_MEASP_LPPDE_ln)) + logsumexp(pooled_MEASP_LPPDE_ln.to_numpy(), axis=0),index=evid)
    cols_ln['EAlnSTWI_%s' % period] = pd.Series(-np.log(len(pooled_MEASTWI_LPPDE_ln)) + logsumexp(pooled_MEASTWI_LPPDE_ln.to_numpy(), axis=0),index=evid)
    cols_ln['EAlnTWIGEO_%s' % period] = pd.Series(-np.log(len(pooled_MEATWIGEO_LPPDE_ln)) + logsumexp(pooled_MEATWIGEO_LPPDE_ln.to_numpy(), axis=0),index=evid)
    cols_ln['EAlnSZT_%s' % period] = pd.Series(-np.log(len(pooled_MEASZT_LPPDE_ln)) + logsumexp(pooled_MEASZT_LPPDE_ln.to_numpy(), axis=0),index=evid)

PO_LPPDE_ln = pd.DataFrame(cols_ln)
PO_LPPDE_ln.index.name = 'evid'
PO_LPPDE_ln = PO_LPPDE_ln.reset_index()

# ensure correct ordering
order = LPPDES['evid'].astype(str)
PO_LPPDE_ln['evid'] = PO_LPPDE_ln['evid'].astype(str)
PO_LPPDE_ln = PO_LPPDE_ln.set_index('evid').loc[order].reset_index()

# 2.0 Feature Selection

## 2.1 Correlation Matrix of features

In [ ]:
cols_of_interest = ['vs30', 'Z1.0', 'Z2.5', 'TRI', 'TWI', 'precip', 'geoage', 'k0', 'PGA','pSA_0.06' ,'pSA_0.1', 'pSA_0.3', 'pSA_0.6', 'pSA_1.0', 'pSA_3.0', 'pSA_6.0']

df_corr = im_dwbar_t6[cols_of_interest]
corr = df_corr.corr()

# Create upper triangle mask
mask = np.triu(np.ones_like(corr, dtype=bool))

# Plot
plt.figure(figsize=(10, 6), dpi = 200)
heatmap = sns.heatmap(
    corr,
    mask=mask,
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 12},
    linewidths=0.5,
    cbar_kws={"shrink": 0.75,
              "pad": 0.005}
)
# plt.title('Correlation Matrix', fontsize=14)
cbar = heatmap.collections[0].colorbar
cbar.set_label(r'Pearson Correlation Coefficient ($\rho$)', fontsize=14)
# Make x and y tick labels bigger
heatmap.set_xticklabels(heatmap.get_xticklabels(), fontsize=14)
heatmap.set_yticklabels(heatmap.get_yticklabels(), fontsize=14)

# Make colourbar tick labels bigger
cbar.ax.tick_params(labelsize=12)
plt.tight_layout()
plt.show()

## 6.2 Principal Component Analysis (PCA)

In [ ]:
cols_of_interest = ['vs30', 'Z1.0', 'Z2.5', 'TRI', 'k0', 'TWI', 'precip','geoage']
X = im_dwbar_t6[cols_of_interest].dropna()

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# How much variance each component explains?
explained = pca.explained_variance_ratio_
print("Explained variance per component:", explained)

# What are the loadings?
loadings = pd.DataFrame(pca.components_.T, index=cols_of_interest, columns=[f'PC{i+1}' for i in range(len(cols_of_interest))])

# Project to 2D
X_pca_2d = X_pca[:, :2]

fig, ax = plt.subplots(figsize=(10,10), dpi = 200)
ax.scatter(X_pca_2d[:,0], X_pca_2d[:,1], alpha=0.5)

for i, var in enumerate(cols_of_interest):
    ax.arrow(0,0, loadings.iloc[i,0]*3.4, loadings.iloc[i,1]*3.4,
              color='k', alpha=0.7, head_width=0.1)
    ax.text(loadings.iloc[i,0]*3.9, loadings.iloc[i,1]*3.9, var, color='k', fontsize = 12)

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_xlabel('Principal Component 1', fontsize=14)
ax.set_ylabel('Principal Component 2', fontsize=14)
ax.set_title('Pincipal Component Analysis Biplot', fontsize=14)
ax.grid()
plt.show()

## 2.3 Canonical Component Analysis (CCA)

### 2.3.1 Canonical Variance

In [ ]:
# Predictor columns
X_cols = ['vs30', 'Z1.0', 'Z2.5', 'TRI', 'precip', 'TWI']

# Response columns
Y_cols = ['PGA', 'pSA_0.06', 'pSA_0.1', 'pSA_1.0', 'pSA_3.0']

# Drop rows with any NaNs in predictors or responses
df = im_dwbar_t6[X_cols + Y_cols].dropna()

X = df[X_cols].values
Y = df[Y_cols].values

# Standardize both X and Y
scaler_X = StandardScaler()
scaler_Y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
Y_scaled = scaler_Y.fit_transform(Y)

cca = CCA(n_components=2)
X_c, Y_c = cca.fit_transform(X_scaled, Y_scaled)

corrs = [np.corrcoef(X_c[:,i], Y_c[:,i])[0,1] for i in range(X_c.shape[1])]
print("Canonical correlations:", corrs)

loadings_X = pd.DataFrame(
    cca.x_weights_,
    index=X_cols,
    columns=[f'CanVar{i+1}' for i in range(X_c.shape[1])]
)
print("\nPredictor loadings:\n", loadings_X)

loadings_Y = pd.DataFrame(
    cca.y_weights_,
    index=Y_cols,
    columns=[f'CanVar{i+1}' for i in range(Y_c.shape[1])]
)
print("\nResponse loadings:\n", loadings_Y)

fig, ax = plt.subplots(1,2, figsize = (12,6), dpi = 200, constrained_layout = True)
ax[0].scatter(X_c[:,0], Y_c[:,0], alpha=0.4)
ax[1].scatter(X_c[:,1], Y_c[:,1], alpha=0.4)
for i in range(2):
    ax[i].set_xlabel('Canonical Variate %.0f (Predictors)'%(i+1))
    ax[i].set_ylabel('Canonical Variate %.0f (Responses)'%(i+1))
    ax[0].set_title(f'First Canonical Correlation: {corrs[0]:.3f}')
    ax[1].set_title(f'Second Canonical Correlation: {corrs[1]:.3f}')
    ax[i].grid(True)
    ax[i].set_xlim(-3,3)
    ax[i].set_ylim(-3,3)
plt.show()

### 2.3.2 Projected Canonical Variance

In [ ]:
X_df = im_dwbar_t6[['vs30', 'Z1.0', 'Z2.5', 'TRI', 'precip']].dropna()
Y_df = im_dwbar_t6[['pSA_0.03', 'pSA_0.1', 'pSA_1.0', 'pSA_3.0']].dropna()

# Align rows
X_df, Y_df = X_df.align(Y_df, join='inner', axis=0)

# Standardize
X_scaled = StandardScaler().fit_transform(X_df)
Y_scaled = StandardScaler().fit_transform(Y_df)

# CCA
cca = CCA(n_components=2)
X_c, Y_c = cca.fit_transform(X_scaled, Y_scaled)

# Loadings (structure coefficients)
X_loadings = pd.DataFrame(np.corrcoef(X_scaled.T, Y_c.T)[0:X_df.shape[1], X_df.shape[1]:],
                          index=X_df.columns, columns=['CanVar1', 'CanVar2'])
Y_loadings = pd.DataFrame(np.corrcoef(Y_scaled.T, X_c.T)[0:Y_df.shape[1], Y_df.shape[1]:],
                          index=Y_df.columns, columns=['CanVar1', 'CanVar2'])
fig, ax = plt.subplots(figsize=(10, 10), dpi=200)
ax.scatter(X_c[:, 0], X_c[:, 1], alpha=0.5, label='Predictor Scores')
ax.scatter(Y_c[:, 0], Y_c[:, 1], alpha=0.5, label='Response Scores', marker='x')

arrow_scale = 6.8       # Longer arrows
text_offset = 1.1       # Push text slightly beyond arrow tip

# Predictor loadings
for var in X_loadings.index:
    x = X_loadings.loc[var, 'CanVar1'] * arrow_scale
    y = X_loadings.loc[var, 'CanVar2'] * arrow_scale
    ax.arrow(0, 0, x, y, color='blue', alpha=0.7, head_width=0.10, length_includes_head=True)
    ax.text(x * text_offset, y * text_offset, var, color='blue', fontsize=12, ha='center', va='center')

# Response loadings
for var in Y_loadings.index:
    x = Y_loadings.loc[var, 'CanVar1'] * arrow_scale
    y = Y_loadings.loc[var, 'CanVar2'] * arrow_scale
    ax.arrow(0, 0, x, y, color='black', alpha=0.7, head_width=0.10, length_includes_head=True)
    ax.text(x * text_offset, y * text_offset, var, color='black', fontsize=12, ha='center', va='center')

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_xlabel('Canonical Variable 1', fontsize=14)
ax.set_ylabel('Canonical Variable 2', fontsize=14)
ax.set_title('Canonical Correlation Analysis Biplot', fontsize=14)
ax.grid()
ax.legend()
plt.show()

# 3.0 Kernel Overview

## 3.1 Kernel Definitions

In [ ]:
# compute azimuth and epicentral distance
def LongLatToPolar(Xst, Xeq):
    geod = Geod(ellps='WGS84')
    r = geod.inv(lons1 = Xeq[:, 0], lats1 = Xeq[:, 1],
        lons2 = Xst[:, 0], lats2 = Xst[:, 1])
    repi = r[2]/1000
    az = r[0]*np.pi/180
    az[az<0] += 2*np.pi
    Xp = np.vstack([repi, az]).T
    return Xp

def getEucDistance(X):
    '''
    Computes a matrix with the Euclidean distance from
    each input x in X to all other inputs in X.

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values

    Outputs
    ------
    dist: Matrix with distance from each row in X to all
        the other rows (nk x nk)
    '''
    sq_dist = (np.square(np.reshape(X[:,0],[-1,1])) +
        np.square(np.reshape(X[:,0],[1,-1])) -
        2 * (np.reshape(X[:,0],[-1,1]) * np.reshape(X[:,0],[1,-1])) *
        np.cos(np.abs(np.reshape(X[:,1],[1,-1]) - np.reshape(X[:,1],[-1,1]) )) )
    sq_dist = np.clip(sq_dist, 0, np.inf)
    dist = np.sqrt(sq_dist)
    dist[np.diag_indices_from(dist)] = 0.0
    return dist

def getAngDistance(X):
    '''
    Computes a matrix with the angular distance from
    each input x in X to all other inputs in X.

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values

    Outputs
    ------
    dist: Matrix with Angular distance from each
        row in X to all the other rows (nk x nk)
    '''
    cos_angle = np.cos( np.abs(np.reshape(X[:,1],[1,-1]) -
                                np.reshape(X[:,1],[-1,1]) ))
    dist =  np.arccos(np.clip(cos_angle, -1, 1))
    dist[np.diag_indices_from(dist)] = 0.0
    return dist

def getSoilDissimilarity(X):
    '''
    Computes a matrix with the soil dissimilarity from
    each input x in X to all other inputs in X.

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values

    Outputs
    ------
    dist: Matrix with soil dissimilarities from each
        row in X to all the other rows (nk x nk)
    '''
    sq_dist = np.square(np.reshape(X[:,2], [-1,1]) - np.reshape(X[:,2], [1,-1]))
    sq_dist = np.clip(sq_dist, 0, np.inf)
    dist = np.sqrt(sq_dist)
    dist[np.diag_indices_from(dist)] = 0.0
    return dist

def getBasinDissimilarity(X):
    '''
    Computes a matrix with the basin dissimilarity from
    each input x in X to all other inputs in X.

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values

    Outputs
    ------
    dist: Matrix with soil dissimilarities from each
        row in X to all the other rows (nk x nk)
    '''
    sq_dist = np.square(np.reshape(X[:,3], [-1,1]) - np.reshape(X[:,3], [1,-1]))
    sq_dist = np.clip(sq_dist, 0, np.inf)
    dist = np.sqrt(sq_dist)
    dist[np.diag_indices_from(dist)] = 0.0
    return dist

def getTopoDissimilarity(X):
    '''
    Computes a matrix with the basin dissimilarity from
    each input x in X to all other inputs in X.

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values

    Outputs
    ------
    dist: Matrix with soil dissimilarities from each
        row in X to all the other rows (nk x nk)
    '''
    sq_dist = np.square(np.reshape(X[:,4], [-1,1]) - np.reshape(X[:,4], [1,-1]))
    sq_dist = np.clip(sq_dist, 0, np.inf)
    dist = np.sqrt(sq_dist)
    dist[np.diag_indices_from(dist)] = 0.0
    return dist

def getRrupDissimilarity(X):
    '''
    Computes a matrix with the basin dissimilarity from
    each input x in X to all other inputs in X.

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values
        Column 5: Rupture distance in km

    Outputs
    ------
    dist: Matrix with soil dissimilarities from each
        row in X to all the other rows (nk x nk)
    '''
    sq_dist = np.square(np.reshape(X[:,3], [-1,1]) - np.reshape(X[:,3], [1,-1]))
    sq_dist = np.clip(sq_dist, 0, np.inf)
    dist = np.sqrt(sq_dist)
    dist[np.diag_indices_from(dist)] = 0.0
    return dist

def rhoE(X, LE, gammaE):
    '''
    Computes the correlation matrix using the
    isotropic model E conditional on parameters
    (LE, gammaE)

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
    LE: Euclidean lengthscale in km
    gammaE: Exponent

    Outputs
    ------
    K: Correlation matrix (nk x nk)
    '''
    distE = getEucDistance(X)
    K = np.exp(- np.power(distE/LE, gammaE))
    return K

def rhoEA(X, LE, gammaE, LA):
    '''
    Computes the correlation matrix using the
    model EA conditional on parameters
    (LE, gammaE, LA)

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
    LE: Euclidean lengthscale in km
    gammaE: Exponent
    LA: Angular lengthscale in degrees

    Outputs
    ------
    K: Correlation matrix (nk x nk)
    '''
    distE = getEucDistance(X)
    KE = np.exp(- np.power(distE/LE, gammaE))

    # distA = getAngDistance(X)
    # distAdeg = distA * 180/np.pi
    # KA = (1 + distAdeg/LA) * np.power(1 - distAdeg/180, 180/LA)

    distA = getAngDistance(X)
    distAdeg = distA * 180.0 / np.pi
    df = np.abs(((distAdeg + 90.0) % 180.0) - 90.0)
    base = np.clip(1.0 - df / 90.0, 0.0, 1.0)
    KA = (1.0 + df / LA) * np.power(base, 90.0 / LA)

    K = KE * KA
    return K

def rhoEAS(X, LE, gammaE, LA, LS, w):
    '''
    Computes the correlation matrix using the
    model EAS conditional on parameters
    (LE, gammaE, LA, LS, w)

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
    LE: Euclidean lengthscale in km
    gammaE: Exponent
    LA: Angular lengthscale in degrees
    LS: Vs30 lengthscale in m/s
    w: weight parameter

    Outputs
    ------
    K: Correlation matrix (nk x nk)
    '''
    distE = getEucDistance(X)
    KE = np.exp(- np.power(distE/LE, gammaE))

    distA = getAngDistance(X)
    distAdeg = distA * 180/np.pi
    KA = (1 + distAdeg/LA) * np.power(1 - distAdeg/180, 180/LA)

    distS = getSoilDissimilarity(X)
    KS = np.exp(- distS/LS)

    K = KE * (w*KA + (1-w)*KS)
    return K

def rhoEASZ(X, LE, gammaE, LA, LS, LZ, w, wS):
    '''
    Computes the correlation matrix using the
    model EAS conditional on parameters
    (LE, gammaE, LA, LS, LZ,  w, wS)

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
    LE: Euclidean lengthscale in km
    gammaE: Exponent
    LA: Angular lengthscale in degrees
    LS: Vs30 lengthscale in m/s
    LZ: Z1.0 lengthscale in m
    w: weight parameter of site term
    wS: weight parameter of Vs30

    Outputs
    ------
    K: Correlation matrix (nk x nk)
    '''
    distE = getEucDistance(X)
    KE = np.exp(- np.power(distE/LE, gammaE))

    distA = getAngDistance(X)
    distAdeg = distA * 180/np.pi
    KA = (1 + distAdeg/LA) * np.power(1 - distAdeg/180, 180/LA)

    distS = getSoilDissimilarity(X)
    KS = np.exp(- distS/LS)

    distZ = getBasinDissimilarity(X)
    KZ = np.exp(- distZ/LZ)

    KS_w = KS * wS
    KZ_w = KZ * (1-wS)

    K = KE * (w * KA + (1-w) * (KS_w + KZ_w))
    return K

def rhoEAST(X, LE, gammaE, LA, LS, LT, w, wS):
    '''
    Computes the correlation matrix using the
    model EAS conditional on parameters
    (LE, gammaE, LA, LS, LZ,  w, wS)

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values in m
    LE: Euclidean lengthscale in km
    gammaE: Exponent
    LA: Angular lengthscale in degrees
    LS: Vs30 lengthscale in m/s
    LT: TRI lengthscale in m
    w: weight parameter of site term
    wS: weight parameter of Vs30

    Outputs
    ------
    K: Correlation matrix (nk x nk)
    '''
    distE = getEucDistance(X)
    KE = np.exp(- np.power(distE/LE, gammaE))

    distA = getAngDistance(X)
    distAdeg = distA * 180/np.pi
    KA = (1 + distAdeg/LA) * np.power(1 - distAdeg/180, 180/LA)

    distS = getSoilDissimilarity(X)
    KS = np.exp(- distS/LS)

    distT = getTopoDissimilarity(X)
    KT = np.exp(- distT/LT)

    KS_w = KS * wS
    KT_w = KT * (1-wS)

    K = KE * (w * KA + (1 - w) * (KS_w + KT_w))
    return K

def rhoEASZT(X, LE, gammaE, LA, LS, LZ, LT, w, wS, wZ, wT):

    '''
    Computes the correlation matrix using the
    model EAS conditional on parameters
    (LE, gammaE, LA, LS, LZ,  w, wS)

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values in m
    LE: Euclidean lengthscale in km
    gammaE: Exponent
    LA: Angular lengthscale in degrees
    LS: Vs30 lengthscale in m/s
    LZ: Z1.0 lengthscale in m
    LT: TRI lengthscale in m
    w: weight parameter of site term
    wS: weight parameter of Vs30
    wZ: weight parameter of Z1.0
    wT: weight parameters of TRI
    Outputs
    ------
    K: Correlation matrix (nk x nk)
    '''
    distE = getEucDistance(X)
    KE = np.exp(- np.power(distE/LE, gammaE))

    distA = getAngDistance(X)
    distAdeg = distA * 180/np.pi
    KA = (1 + distAdeg/LA) * np.power(1 - distAdeg/180, 180/LA)

    distS = getSoilDissimilarity(X)
    KS = np.exp(- distS/LS)

    distZ = getBasinDissimilarity(X)
    KZ = np.exp(- distZ/LZ)

    distT = getTopoDissimilarity(X)
    KT = np.exp(- distT/LT)

    KS_w = KS * wS
    KZ_w = KZ * wZ
    KT_w = KT * wT

    K = KE * (w * KA + (1 - w) * (KS_w + KZ_w + KT_w))
    return K

def rhoEAR(X, LE, gammaE, LA, LR, w):

    '''
    Computes the correlation matrix using the
    model EAS conditional on parameters
    (LE, gammaE, LA, LS, LZ,  w, wS)

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values in m
        column 5: Rrup values in km
    LE: Euclidean lengthscale in km
    gammaE: Exponent
    LA: Angular lengthscale in degrees
    LR: Rupture distance length-scale in km
    w: weight parameter of angular term
    Outputs
    ------
    K: Correlation matrix (nk x nk)
    '''
    distE = getEucDistance(X)
    KE = np.exp(- np.power(distE/LE, gammaE))

    distA = getAngDistance(X)
    distAdeg = distA * 180/np.pi
    KA = (1 + distAdeg/LA) * np.power(1 - distAdeg/180, 180/LA)

    distR = getRrupDissimilarity(X)
    KR = np.exp(-distR/LR)

    K = KE * (w*KA+(1-w)*KR)
    return K

def rhoEARS(X, LE, gammaE, LA, LR, LS, wA, wS):

    '''
    Computes the correlation matrix using the
    model EAS conditional on parameters
    (LE, gammaE, LA, LS, LZ,  w, wS)

    Inputs
    ------
    X: Input matrix (nk x 3)
        Column 0: Epicentral distance in km
        Column 1: Epicentral azimuth in radians
        Column 2: Vs30 values in m/s
        Column 3: Z1.0 values in m
        Column 4: TRI values in m
        column 5: Rrup values in km
    LE: Euclidean lengthscale in km
    gammaE: Exponent
    LA: Angular lengthscale in degrees
    LR: Rupture distance length-scale in km
    w: weight parameter of angular term
    Outputs
    ------
    K: Correlation matrix (nk x nk)
    '''
    distE = getEucDistance(X)
    KE = np.exp(- np.power(distE/LE, gammaE))

    # distA = getAngDistance(X)
    # distAdeg = distA * 180/np.pi
    # KA = (1 + distAdeg/LA) * np.power(1 - distAdeg/180, 180/LA)
    distA = getAngDistance(X)
    distAdeg = distA * 180.0 / np.pi
    df = np.abs(((distAdeg + 90.0) % 180.0) - 90.0)
    base = np.clip(1.0 - df / 90.0, 0.0, 1.0)
    KA = (1.0 + df / LA) * np.power(base, 90.0 / LA)

    distS = getSoilDissimilarity(X)
    KS = np.exp(- distS/LS)

    distR = getRrupDissimilarity(X)
    KR = np.exp(-distR/LR)

    K = KE * ((wA*KA+(1-wA)*KR)*(1-wS) + wS*KS)
    return K

## 3.2 Bodenmann et al., (2023) Kernals

In [ ]:
# load fictitious soil conditions (from Bodenmann et al., (2023) examples)
df = pd.read_csv(SCM_DIR / "intro_virtualregion.csv")

# define model length scales
psi_E = dict(LE=30.0, gammaE=1.0)
psi_EA = dict(LE=30.0, gammaE=1.0, LA=45.0)
psi_EAS = dict(LE=30.0, gammaE=1.0, LA=18.0, LS=100.0, w=0.50)

# Store input matrix X
X = df[['epi_dist', 'epi_azimuth', 'vs30']].values

# set event location
x_event, y_event = (-5, 10)

# index of reference site
ref_idx = 1990

# Compute correlations
kE = rhoE(X, **psi_E)[ref_idx, :]
kEA = rhoEA(X, **psi_EA)[ref_idx, :]
kEAS = rhoEAS(X, **psi_EAS)[ref_idx, :]

# define grid
grid_shape = (int(np.sqrt(len(df))), int(np.sqrt(len(df))))
Xcgrid = df['x'].values.reshape(grid_shape)
Ycgrid = df['y'].values.reshape(grid_shape)
vals = [df.vs30.values.reshape(grid_shape),
        kE.reshape(grid_shape), kEA.reshape(grid_shape),
        kEAS.reshape(grid_shape)]
lev = np.arange(0,1.2,0.2)
x_ref, y_ref = (df.x[ref_idx], df.y[ref_idx])

# plot
fig, ax = plt.subplots(1,4, figsize = (12,3.5), dpi = 300, constrained_layout = True)
# vs30 values grid
imvs30 = ax[0].pcolormesh(Xcgrid, Ycgrid, vals[0],
                vmin=100, vmax=800, cmap=plt.get_cmap('Greens'),
                shading='gouraud',alpha=1)

# Euclidean difference distance grid
imE = ax[1].contourf(Xcgrid, Ycgrid, vals[1],
                vmin=0, vmax=1, cmap=plt.get_cmap('Blues'),
                levels=lev, alpha=0.8)

# auglar difference distance grid
imEA = ax[2].contourf(Xcgrid, Ycgrid, vals[2],
                vmin=0, vmax=1, cmap=plt.get_cmap('Blues'),
                levels=lev, alpha=0.8)

# vs30 difference distance grid
imEAS = ax[3].contourf(Xcgrid, Ycgrid, vals[3],
                vmin=0, vmax=1, cmap=plt.get_cmap('Blues'),
                levels=lev, alpha=0.8)

# plot reference point and epicenter
ax[0].set_ylabel('Northing [km]')
for i in range(4):
    ax[i].set_aspect('equal')
    ax[i].set_xlabel('Easting [km]')
    ax[i].scatter(x_ref, y_ref, marker='x', color='black', s=30, alpha=1, lw=1.5)
    ax[i].scatter(x_event, y_event, marker='*', edgecolor='black', color='none', s=50, lw=1.5)

# annotate
ax[0].text(-5, 6.5, 'epicenter', fontsize=8, ha='center')
ax[0].annotate("reference site",
                        xy=(x_ref, y_ref-0.5), xycoords='data',
                        xytext=(x_ref, -11.3), textcoords='data',
                        arrowprops=dict(arrowstyle="->",
                                        connectionstyle="arc3", lw=0.8),
                        fontsize=8, ha='center',va='top')

# colorbars - need to ensure same width as subplot
sm = plt.cm.ScalarMappable(cmap=plt.get_cmap('Greens'),
        norm=plt.Normalize(vmin=100, vmax=800))
cax1 = inset_axes(ax[0],
                  width="100%", height="6%",
                  loc="lower center",
                  bbox_to_anchor=(0, -0.30, 1, 1),
                  bbox_transform=ax[0].transAxes,
                  borderpad=0)
cbar = fig.colorbar(sm, cax=cax1, orientation='horizontal')
cbar.set_label('$v_{s30}$ [m/s]')

cax2 = inset_axes(ax[2],
                  width="100%", height="6%",
                  loc="lower center",
                  bbox_to_anchor=(0, -0.30, 1, 1),
                  bbox_transform=ax[2].transAxes,
                  borderpad=0)
cbar2 = fig.colorbar(imEA, cax=cax2, orientation='horizontal')
cbar2.set_label('Correlation')
plt.show()

## 3.3 Extended Kernels (Rampersad et al., 2026)

### 3.3.1 Distance Function: Rrup, Repi, Raz

#### 3.3.1.1 Corners of CMT

In [ ]:
def fault_rectangle_centered_xy(
    x_km, y_km, depth_km, strike_deg, dip_deg, length_km, width_km,
    clamp_top_to_surface=False
):
    """
    Build a planar rectangular fault patch centered at (x_km, y_km, depth_km)
    in a local Cartesian system (x east, y north, units = km).

    Inputs:
      - length_km: along-strike length on the fault plane
      - width_km:  down-dip width on the fault plane
      - strike_deg: azimuth of the fault trace (clockwise from north)
      - dip_deg: dip angle (0 = horizontal, 90 = vertical)
      - depth_km: hypocenter depth (positive down)
      - clamp_top_to_surface: if True, any up-dip corner <0 km depth is clamped to 0

    Returns (list of 5 tuples):
      [(x, y, depth_km) ...] ordered:
      updip-left, updip-right, downdip-right, downdip-left, repeat first
    """

    # Half-dimensions
    L2 = length_km / 2.0
    W2 = width_km / 2.0

    # Horizontal and vertical offsets for half-width
    horiz_dip_km = W2 * np.cos(np.radians(dip_deg))
    vert_dip_km  = W2 * np.sin(np.radians(dip_deg))

    # Strike unit vector (along fault trace, clockwise from north)
    strike_rad = np.radians(strike_deg)
    sx = np.sin(strike_rad)   # east component
    sy = np.cos(strike_rad)   # north component

    # Perpendicular (to the right of strike)
    px =  sy
    py = -sx

    # Along-strike endpoints of the centerline
    x_L = x_km - L2 * sx
    y_L = y_km - L2 * sy
    x_R = x_km + L2 * sx
    y_R = y_km + L2 * sy

    # Move updip (negative perpendicular) and downdip (positive perpendicular)
    # Left end
    ul_x = x_L - horiz_dip_km * px
    ul_y = y_L - horiz_dip_km * py
    dl_x = x_L + horiz_dip_km * px
    dl_y = y_L + horiz_dip_km * py
    ul_dep = max(0.0, depth_km - vert_dip_km) if clamp_top_to_surface else (depth_km - vert_dip_km)
    dl_dep = depth_km + vert_dip_km

    # Right end
    ur_x = x_R - horiz_dip_km * px
    ur_y = y_R - horiz_dip_km * py
    dr_x = x_R + horiz_dip_km * px
    dr_y = y_R + horiz_dip_km * py
    ur_dep = max(0.0, depth_km - vert_dip_km) if clamp_top_to_surface else (depth_km - vert_dip_km)
    dr_dep = depth_km + vert_dip_km

    corners = [
        (ul_x, ul_y, ul_dep),
        (ur_x, ur_y, ur_dep),
        (dr_x, dr_y, dr_dep),
        (dl_x, dl_y, dl_dep),
        (ul_x, ul_y, ul_dep)
    ]
    return corners

# discretize_nodel/fault_plane
def discretize_planes_xy(corners, points_per_km=10):
    """
    Discretize a rectangular fault plane given in local Cartesian coordinates.

    Parameters
    ----------
    corners : array-like, shape (4,3) or (5,3)
        Fault corners as (x_km, y_km, depth_km). Depth positive down.
        If 5 points are given with closure, the last is dropped.
    points_per_km : float
        Sampling density along strike and dip.

    Returns
    -------
    xs, ys, zs : flat arrays
        Discretized coordinates (km).
    """
    C = np.asarray(corners, float).squeeze()
    if C.shape[0] == 5 and np.allclose(C[0], C[-1]):
        C = C[:-1]
    if C.shape != (4, 3):
        raise ValueError(f"Expected 4x3 corners, got {C.shape}")

    xyz = C.copy()

    # --- enforce consistent ordering (convex hull + clockwise) ---
    cx, cy = xyz[:, 0].mean(), xyz[:, 1].mean()
    angles = np.arctan2(xyz[:, 1] - cy, xyz[:, 0] - cx)
    order = np.argsort(angles)
    xyz = xyz[order]

    # assign: top edge = shallowest two adjacent points
    depths = xyz[:, 2]
    shallow_idx = np.argsort(depths)[:2]
    if (np.abs(shallow_idx[0] - shallow_idx[1]) not in [1, 3]):
        xyz = np.roll(xyz, -shallow_idx[0], axis=0)

    top = xyz[0:2]
    bot = xyz[2:4]

    # order left﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿right along strike
    v_strike = top[1, :2] - top[0, :2]
    u = v_strike / np.linalg.norm(v_strike)
    top = top[np.argsort(top[:, :2] @ u)]
    bot = bot[np.argsort(bot[:, :2] @ u)]

    # lengths (km)
    Ls = 0.5 * (
        np.linalg.norm(top[1, :2] - top[0, :2]) +
        np.linalg.norm(bot[1, :2] - bot[0, :2])
    )
    Ld = 0.5 * (
        np.linalg.norm(bot[0, :2] - top[0, :2]) +
        np.linalg.norm(bot[1, :2] - top[1, :2])
    )

    nstrike = max(1, int(np.ceil(Ls * points_per_km)))
    ndip = max(1, int(np.ceil(Ld * points_per_km)))

    s = np.linspace(0.0, 1.0, nstrike + 1)
    t = np.linspace(0.0, 1.0, ndip + 1)

    xs, ys, zs = [], [], []

    # edges along strike
    top_edge_xy = (1 - s)[:, None] * top[0, :2] + s[:, None] * top[1, :2]
    top_edge_z = (1 - s) * top[0, 2] + s * top[1, 2]
    bot_edge_xy = (1 - s)[:, None] * bot[0, :2] + s[:, None] * bot[1, :2]
    bot_edge_z = (1 - s) * bot[0, 2] + s * bot[1, 2]

    # bilinear fill
    for tt in t:
        row_xy = (1 - tt) * top_edge_xy + tt * bot_edge_xy
        row_z = (1 - tt) * top_edge_z + tt * bot_edge_z
        xs.append(row_xy[:, 0])
        ys.append(row_xy[:, 1])
        zs.append(row_z)

    return np.concatenate(xs), np.concatenate(ys), np.concatenate(zs)

#### 3.3.1.2 Rupture Distance, Rrup

In [ ]:
def compute_rrup_rjb_corners_xy(
    site_x, site_y, site_depths_km,
    rup_x, rup_y, rup_depths_km,
    rupture_corners=None,        # (4,3) or (4,3,N) in (x_km, y_km, depth_km)
    site_block=4000, rup_block=4000,
    show_progress=True
):
    """
    Chunked Euclidean (Cartesian) based distances in km:
      Rjb  = min horizontal distance to rupture cloud (km), then 0 inside quad(s).
      Rrup = min sqrt(H^2 + V^2) over rupture points (km), computed in chunks.

    Inputs are already in a local Cartesian system (x east, y north, km).
    """

    N = site_x.size
    R = rup_x.size

    rrup_min = np.full(N, np.inf, dtype=float)
    rjb_min  = np.full(N, np.inf, dtype=float)

    # --- progress bars ---
    site_iter = range(0, N, site_block)
    if show_progress:
        site_iter = tqdm(site_iter, total=math.ceil(N/site_block), desc="Sites", unit="blk")
    n_rchunks = math.ceil(R / rup_block)
    inner_bar = tqdm(
        total=n_rchunks, desc="  Rupture chunks", unit="ch",
        position=1, dynamic_ncols=True,
        disable=not show_progress
    )

    # --- main loops ---
    for i0 in site_iter:
        i1 = min(i0 + site_block, N)
        xb = site_x[i0:i1]
        yb = site_y[i0:i1]
        db = site_depths_km[i0:i1]  # (S,)

        # rolling minima for this site block
        rjb_b      = np.full(xb.size, np.inf, dtype=float)
        rrup_b2min = np.full(xb.size, np.inf, dtype=float)

        if show_progress:
            inner_bar.reset()
            inner_bar.set_postfix_str(f"sites {i0}:{i1-1}")
            inner_bar.refresh()

        for j0 in range(0, R, rup_block):
            j1 = min(j0 + rup_block, R)
            rx = rup_x[j0:j1]
            ry = rup_y[j0:j1]
            rd = rup_depths_km[j0:j1]

            # horizontal Euclidean distances (km), shape (S, Cr)
            dx = xb[:, None] - rx[None, :]
            dy = yb[:, None] - ry[None, :]
            H = np.sqrt(dx*dx + dy*dy)

            # vertical differences (km)
            V = db[:, None] - rd[None, :]

            # update minima
            rjb_b       = np.minimum(rjb_b,  np.min(H, axis=1))
            rrup_b2min  = np.minimum(rrup_b2min, np.min(H*H + V*V, axis=1))

            if show_progress:
                inner_bar.update(1)

        # write back
        rjb_min[i0:i1]  = rjb_b
        rrup_min[i0:i1] = np.sqrt(rrup_b2min)

    inner_bar.close()

    # --- Optional: set Rjb=0 if inside any surface projection (x/y polygon check) ---
    if rupture_corners is not None:
        C = np.asarray(rupture_corners, float).squeeze()
        planes = [C] if C.ndim == 2 else [C[:, :, i] for i in range(C.shape[2])]
        pts = np.column_stack([site_x, site_y])

        plane_iter = planes
        if show_progress:
            plane_iter = tqdm(plane_iter, desc="Inside test (polys)", unit="poly", position=1)

        inside_any = np.zeros(N, dtype=bool)
        for P in plane_iter:
            # order the 4 corners roughly CCW to avoid self-crossing
            cx, cy = np.mean(P[:,0]), np.mean(P[:,1])
            ang = np.arctan2(P[:,1] - cy, P[:,0] - cx)
            poly = MplPath(np.column_stack([P[np.argsort(ang),0], P[np.argsort(ang),1]]), closed=True)
            inside_any |= poly.contains_points(pts)

        rjb_min[inside_any] = 0.0

    return rrup_min, rjb_min

#### 3.3.1.3 Azmithal and Epicentral Distances

In [ ]:
def XYtoPolar(Xst, Xeq):
    """
    Compute epicentral distance and azimuth in a local Cartesian system.

    Parameters
    ----------
    Xst : array-like, shape (N,2)
        Station coordinates (x_km, y_km).
    Xeq : array-like, shape (N,2) or (1,2)
        Earthquake epicenter coordinates (x_km, y_km).
        If shape (1,2), broadcast to all stations.

    Returns
    -------
    Xp : array, shape (N,2)
        Columns: [repi_km, azimuth_rad]
        - repi_km: epicentral distance (km)
        - azimuth_rad: azimuth clockwise from north, in [0, 2﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿)
    """
    Xst = np.asarray(Xst, float)
    Xeq = np.asarray(Xeq, float)

    # Broadcast epicenter if single point
    if Xeq.ndim == 1:
        Xeq = np.tile(Xeq, (Xst.shape[0], 1))

    dx = Xst[:, 0] - Xeq[:, 0]
    dy = Xst[:, 1] - Xeq[:, 1]

    repi = np.sqrt(dx*dx + dy*dy)

    # azimuth: clockwise from north
    az = np.arctan2(dx, dy)   # note: arctan2(y,x) but here we want CW from north
    az[az < 0] += 2*np.pi

    return np.column_stack([repi, az])

### 3.3.2 Extended Kernel visualization

#### 3.3.2.1 Parameter Computation: Rrup, Rjb, Repi from plane

In [ ]:
# set event location and depth
x_event, y_event, z_event = (-5, 10, 5)

# nodal plane: strike/dip/rake
NP = [88, 70, 111]

# fault dimensions in km i.e., downdip width and along strike length
W = 5
L = 10

# Get fault rectangles
rect = fault_rectangle_centered_xy(
    x_event, y_event, z_event,
    NP[0], NP[1],
    L, W,
    clamp_top_to_surface=False
)
rect_xy = [(lon, lat) for lon, lat, _ in rect]

# discretize rectangle
rup_lons_np1, rup_lats_np1, rup_depths_np1 = discretize_planes_xy(rect, points_per_km=10)
fig, ax = plt.subplots(figsize=(12, 6), dpi = 200)
sc1 = ax.scatter(rup_lons_np1, rup_lats_np1, c=rup_depths_np1,cmap="viridis", s=5)
ax.plot(*zip(*rect_xy), color="red", lw=1.5)  # bounding box outline
ax.set_title("Rect1")
ax.set_xlabel("Easting X")
ax.set_ylabel("Northing Y")
fig.colorbar(sc1, ax=ax, label="Depth (km)")
plt.show()

# source-to-site distances
df['z'] = np.zeros(len(df))
rrup_np1, rjb_np1 = compute_rrup_rjb_corners_xy(
    df.x.values, df.y.values, df.z.values,
    rup_lons_np1, rup_lats_np1, rup_depths_np1,
    rupture_corners=rect
)
df['r_rup'] = rrup_np1
df['r_jb'] = rjb_np1

Xstat = df[['x','y']].values
Xevid = np.zeros_like(Xstat)
Xevid[:,0] = x_event
Xevid[:,1] = y_event
Xp = XYtoPolar(Xstat, Xevid)
df['r_epi'] = Xp[:,0]
df['epi_azimuth'] = Xp[:,1]

#### 3.3.2.2 Kernel Plots

In [ ]:
# define model length scales
psi_E = dict(LE=30.0, gammaE=1.0)
psi_EA = dict(LE=30.0, gammaE=1.0, LA=18.0)
psi_EAS = dict(LE=30.0, gammaE=1.0, LA=18.0, LS=100.0, w=0.50)
psi_EAR = dict(LE=30.0, gammaE=1.0, LA=25.0, LR=20.0, w=0.60)
psi_EARS = dict(LE=30.0, gammaE=1.0, LA=25.0, LR=20.0, LS=100, wA=0.60, wS = 0.3)

# Store input matrix X
X = df[['epi_dist', 'epi_azimuth', 'vs30','r_rup']].values

# index of reference site
ref_idx = 1990

# Compute correlations
kE = rhoE(X, **psi_E)[ref_idx, :]
kEA = rhoEA(X, **psi_EA)[ref_idx, :]
kEAS = rhoEAS(X, **psi_EAS)[ref_idx, :]
kEAR = rhoEAR(X, **psi_EAR)[ref_idx, :]
kEARS = rhoEARS(X, **psi_EARS)[ref_idx, :]

# define grid
grid_shape = (int(np.sqrt(len(df))), int(np.sqrt(len(df))))
Xcgrid = df['x'].values.reshape(grid_shape)
Ycgrid = df['y'].values.reshape(grid_shape)
vals = [df.vs30.values.reshape(grid_shape),
        kE.reshape(grid_shape), kEA.reshape(grid_shape),
        kEAS.reshape(grid_shape), kEAR.reshape(grid_shape), kEARS.reshape(grid_shape)]
lev = np.arange(0,1.2,0.2)
x_ref, y_ref = (df.x[ref_idx], df.y[ref_idx])

# plot
fig, ax = plt.subplots(1,5, figsize = (14,3.5), dpi = 300, constrained_layout = True)
# vs30 values grid
imvs30 = ax[0].pcolormesh(Xcgrid, Ycgrid, vals[0],
                vmin=100, vmax=800, cmap=plt.get_cmap('Greens'),
                shading='gouraud',alpha=1)

# Euclidean difference distance grid
imE = ax[1].contourf(Xcgrid, Ycgrid, vals[1],
                vmin=0, vmax=1, cmap=plt.get_cmap('Blues'),
                levels=lev, alpha=0.8)

# auglar difference distance grid
imEA = ax[2].contourf(Xcgrid, Ycgrid, vals[2],
                vmin=0, vmax=1, cmap=plt.get_cmap('Blues'),
                levels=lev, alpha=0.8)

# Rrup difference distance grid
imEAR = ax[3].contourf(Xcgrid, Ycgrid, vals[4],
                vmin=0, vmax=1, cmap=plt.get_cmap('Blues'),
                levels=lev, alpha=0.8)

# vs30 difference distance grid
imEAS = ax[4].contourf(Xcgrid, Ycgrid, vals[5],
                vmin=0, vmax=1, cmap=plt.get_cmap('Blues'),
                levels=lev, alpha=0.8)

# plot reference point and epicenter
ax[0].set_ylabel('Northing [km]')
for i in range(5):
    ax[i].set_aspect('equal')
    ax[i].set_xlabel('Easting [km]')
    ax[i].scatter(x_ref, y_ref, marker='x', color='black', s=30, alpha=1, lw=1.5)
    ax[i].scatter(x_event, y_event, marker='*', edgecolor='black', color='none', s=50, lw=1.5)
    ax[i].plot(*zip(*rect_xy), '--k', lw=1.5)

# annotate
ax[0].text(-5, 6.5, 'epicenter', fontsize=8, ha='center')
ax[0].annotate("reference site",
                        xy=(x_ref, y_ref-0.5), xycoords='data',
                        xytext=(x_ref, -11.3), textcoords='data',
                        arrowprops=dict(arrowstyle="->",
                                        connectionstyle="arc3", lw=0.8),
                        fontsize=8, ha='center',va='top')


# colorbars - need to ensure same width as subplot
sm = plt.cm.ScalarMappable(cmap=plt.get_cmap('Greens'),
        norm=plt.Normalize(vmin=100, vmax=800))
cax1 = inset_axes(ax[0],
                  width="100%", height="6%",
                  loc="lower center",
                  bbox_to_anchor=(0, -0.30, 1, 1),
                  bbox_transform=ax[0].transAxes,
                  borderpad=0)
cbar = fig.colorbar(sm, cax=cax1, orientation='horizontal')
cbar.set_label('$v_{s30}$ [m/s]')

cax2 = inset_axes(ax[2],
                  width="100%", height="6%",
                  loc="lower center",
                  bbox_to_anchor=(0, -0.30, 1, 1),
                  bbox_transform=ax[2].transAxes,
                  borderpad=0)
cbar2 = fig.colorbar(imEA, cax=cax2, orientation='horizontal')
cbar2.set_label('Correlation')
plt.show()

# 3.0 Model Selection

## 3.1 Model Comparison Plots: Rank and elpd_loo

### 3.1.1 Linear model

In [ ]:
keep_idx = [0, 3, 6, 11]               # PGA, pSA_0.1, pSA_1.0, pSA_6.0
labels    = [pSAobs_T[i] for i in keep_idx]

fig, axes = plt.subplots(1, 4, figsize=(17, 7), dpi =200, constrained_layout=True)

# compact x tick formatter (e.g., -10,250 instead of long floats)
fmt = mtick.FuncFormatter(lambda x, pos: f"{x:,.0f}")
for j, (ax, i, lab) in enumerate(zip(axes, keep_idx, labels)):
    comp_df = model_compare[i].set_index("model")
    comp_df.index = comp_df.index.str.replace("^model", "", regex=True)  # remove 'model' prefix
    comp_df.index.name = None
    az.plot_compare(comp_df, insample_dev=False, ax=ax, plot_ic_diff=True)

    # titles/labels
    ax.set_title(lab, fontsize=14)
    ax.set_xlabel(r"$elpd_{loo}$ (log)", fontsize=14)
    ax.set_ylabel("")

    # cleaner ticks & spacing to avoid overlap
    ax.xaxis.set_major_formatter(fmt)
    ax.xaxis.set_major_locator(mtick.MaxNLocator(4))
    ax.tick_params(axis="x", labelsize=14)
    ax.tick_params(axis="y", labelsize=14)
    ax.grid(True, which = 'both', alpha=0.25)

# axes[0].set_ylabel("Ranked models (Top = Rank 1)", fontsize=14)
axes[0].set_title('PGA', fontsize=14)
axes[1].set_title('SA(0.1s)', fontsize=14)
axes[2].set_title('SA(1.0s)', fontsize=14)
axes[3].set_title('SA(6.0s)', fontsize=14)
axes[3].legend(fontsize=12)
plt.show()


### 3.1.2 Log model

In [ ]:
keep_idx = [0, 3, 6, 11]               # PGA, pSA_0.1, pSA_1.0, pSA_6.0
labels    = [pSAobs_T[i] for i in keep_idx]

fig, axes = plt.subplots(1, 4, figsize=(17, 7), dpi =200, constrained_layout=True)

# compact x tick formatter (e.g., -10,250 instead of long floats)
fmt = mtick.FuncFormatter(lambda x, pos: f"{x:,.0f}")
for j, (ax, i, lab) in enumerate(zip(axes, keep_idx, labels)):
    comp_df = model_compare_ln[i].set_index("model")
    comp_df.index = comp_df.index.str.replace("^model", "", regex=True)  # remove 'model' prefix
    comp_df.index.name = None
    az.plot_compare(comp_df, insample_dev=False, ax=ax, plot_ic_diff=True)

    # titles/labels
    ax.set_title(lab, fontsize=14)
    ax.set_xlabel(r"$elpd_{loo}$ (log)", fontsize=14)
    ax.set_ylabel("")

    # cleaner ticks & spacing to avoid overlap
    ax.xaxis.set_major_formatter(fmt)
    ax.xaxis.set_major_locator(mtick.MaxNLocator(4))
    ax.tick_params(axis="x", labelsize=14)
    ax.tick_params(axis="y", labelsize=14)
    ax.grid(True, which = 'both', alpha=0.25)

# axes[0].set_ylabel("Ranked models (Top = Rank 1)", fontsize=14)
axes[0].set_title('PGA', fontsize=14)
axes[1].set_title('SA(0.1s)', fontsize=14)
axes[2].set_title('SA(1.0s)', fontsize=14)
axes[3].set_title('SA(6.0s)', fontsize=14)
axes[3].legend(fontsize=12)
plt.show()


## 3.2 Model Comparison Plots: p_loo vs. elpd_loo

### 3.2.1 Log models

In [ ]:
# Create 1x4 subplots
fig, axes = plt.subplots(1, 4, figsize=(12, 4), dpi=200, constrained_layout=True)
axes = axes.ravel()

# Pick a consistent colormap so each model keeps the same color across panels
all_models = sorted(set().union(*[model_compare_ln[i]["model"] for i in keep_idx]))
colors = plt.cm.tab20.colors  # or try 'Set2', 'tab20'
color_map = {m.replace("model", ""): colors[k % len(colors)] for k, m in enumerate(all_models)}

# Plot each period
for ax, i, lab in zip(axes, keep_idx, labels):
    df = model_compare_ln[i].copy()
    df = df.set_index("model")
    df.index = df.index.str.replace("^model", "", regex=True)
    df.index.name = None

    # Scatter by model color
    for m, (x, y) in df[["p_loo", "elpd_loo"]].iterrows():
        ax.scatter(x, y, s=60, color=color_map[m], label=m, edgecolors='k', lw = 0.25, alpha=0.7)

    ax.set_xlabel(r"$p_{\mathrm{loo}}$", fontsize=12)
    # ax.set_ylabel(r"$\mathrm{ELPD}_{\mathrm{LOO}}$", fontsize=12)
    ax.set_title(lab, fontsize=12)
    ax.grid(True, alpha=0.3)
# Build one shared legend (deduplicate labels)
handles, labels_unique = [], []
for m in sorted(color_map):
    handles.append(plt.Line2D([], [], color=color_map[m], marker='o', linestyle='', label=m))
    labels_unique.append(m)

axes[0].set_xlim(2,12); axes[1].set_xlim(2,12); axes[2].set_xlim(2,12)
axes[3].set_xlim(2,24)
axes[0].set_title('PGA', fontsize=12)
axes[1].set_title('SA(0.1s)', fontsize=12)
axes[2].set_title('SA(1.0s)', fontsize=12)
axes[3].set_title('SA(6.0s)', fontsize=12)
axes[0].set_ylabel(r"$\mathrm{ELPD}_{\mathrm{LOO}}$", fontsize=12)
axes[0].legend(fontsize=9, loc='lower right')
plt.show()


### 3.2.2 Linear models

In [ ]:
# Create 1x4 subplots
fig, axes = plt.subplots(1, 4, figsize=(12, 4), dpi=200, constrained_layout=True)
axes = axes.ravel()

# Pick a consistent colormap so each model keeps the same color across panels
all_models = sorted(set().union(*[model_compare[i]["model"] for i in keep_idx]))
colors = plt.cm.tab20.colors  # or try 'Set2', 'tab20'
color_map = {m.replace("model", ""): colors[k % len(colors)] for k, m in enumerate(all_models)}

# Plot each period
for ax, i, lab in zip(axes, keep_idx, labels):
    df = model_compare[i].copy()
    df = df.set_index("model")
    df.index = df.index.str.replace("^model", "", regex=True)
    df.index.name = None

    # Scatter by model color
    for m, (x, y) in df[["p_loo", "elpd_loo"]].iterrows():
        ax.scatter(x, y, s=60, color=color_map[m], label=m, edgecolors='k', lw = 0.25, alpha=0.7)

    ax.set_xlabel(r"$p_{\mathrm{loo}}$", fontsize=12)
    # ax.set_ylabel(r"$\mathrm{ELPD}_{\mathrm{LOO}}$", fontsize=12)
    ax.set_title(lab, fontsize=12)
    ax.grid(True, alpha=0.3)
# Build one shared legend (deduplicate labels)
handles, labels_unique = [], []
for m in sorted(color_map):
    handles.append(plt.Line2D([], [], color=color_map[m], marker='o', linestyle='', label=m))
    labels_unique.append(m)

axes[0].set_xlim(2,12); axes[1].set_xlim(2,12); axes[2].set_xlim(2,12)
axes[3].set_xlim(2,24)
axes[0].set_title('PGA', fontsize=12)
axes[1].set_title('SA(0.1s)', fontsize=12)
axes[2].set_title('SA(1.0s)', fontsize=12)
axes[3].set_title('SA(6.0s)', fontsize=12)
axes[0].set_ylabel(r"$\mathrm{ELPD}_{\mathrm{LOO}}$", fontsize=12)
axes[0].legend(fontsize=9, loc='lower right')
plt.show()


# 4.0 Non-stationary SCM Performance

## 4.1 Pooled Model Performance

### 4.1.1 All Models

#### 4.1.1.1 Linear models

In [ ]:
# Load Relative Difference from LPPDP (Performance of Pooled Models)
fig, ax = plt.subplots(figsize=(12, 6), dpi=150, constrained_layout=True)

# Compute differences from baseline model E
diffs = {
        'EA': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEA']['rel_diff'].reset_index(drop=True) -
               LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EAS': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEAS']['rel_diff'].reset_index(drop=True) -
                LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EASZ': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEASZ']['rel_diff'].reset_index(drop=True) -
                 LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EAST': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEAST']['rel_diff'].reset_index(drop=True) -
                 LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EASP': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEASP']['rel_diff'].reset_index(drop=True) -
                 LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EASTWI': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEASTWI']['rel_diff'].reset_index(drop=True) -
                 LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EATWIGEO': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEATWIGEO']['rel_diff'].reset_index(drop=True) -
                 LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EASZT': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEASZT']['rel_diff'].reset_index(drop=True) -
                  LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
}

colors = {
    'EA':      'royalblue',     # strong blue
    'EAS':     'crimson',       # vivid red
    'EASZ':    'seagreen',      # green with blue tint
    'EAST':    'darkorange',    # deep orange
    'EASP':    'mediumvioletred', # magenta-pink
    'EASTWI':  'slateblue',     # purple-blue
    'EATWIGEO':'black',         # neutral black
    'EASZT':   'saddlebrown',   # earthy brown
}

markers = {
    'EA': 's',       # square
    'EAS': 'o',      # circle
    'EASZ': '^',     # triangle up
    'EAST': 'D',     # diamond
    'EASP': 'v',     # triangle down
    'EASTWI': '>',   # triangle right
    'EATWIGEO': 'X', # x-shaped
    'EASZT': '<',    # triangle left
}



# Sort models by final relative difference
sorted_models = sorted(diffs.keys(), key=lambda m: diffs[m].iloc[-1])

# Plot stacked differences
previous_diff = pd.Series([0] * len(pSA_T[:12]))  # Baseline

for model in sorted_models:
        current_diff = diffs[model]
        ax.plot(pSA_T[:12], current_diff, marker=markers[model], linestyle='-', color=colors[model],
                markersize=6, alpha=0.9, label=f'Model {model}')
        ax.fill_between(pSA_T[:12], previous_diff, current_diff, color=colors[model], alpha=0.4)
        previous_diff = current_diff

# Baseline line and annotation
ax.axhline(y=0, xmin=0.001, xmax=6, linestyle='-', color='black', linewidth=3, zorder=0)
ax.text(0.12, 0.2, 'Baseline: LPPD of isotropic model E',
            fontsize=10,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', linewidth=0.8))

# Axis configuration
ax.set_ylabel('Relative LPPD Difference (%)')
ax.set_xlabel('Period T [s]')
ax.set_xscale('log')
ax.set_xlim([0.01, 6])
ax.set_ylim([0,5])
ax.set_ylim(bottom=0)
ax.grid(which='both', linestyle='-', linewidth=0.75, color='gray', alpha=0.2)

# Explicit ticks
ax.set_xticks([0.01, 0.03, 0.06, 0.1, 0.3, 1.0, 3.0, 6.0])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.get_xaxis().set_minor_formatter(plt.NullFormatter())

# Legend
ax.legend(loc='upper left', fontsize=10, frameon=True, framealpha=0.8)

# Title
# fig.suptitle(r'Relative Difference in LPPD', fontsize=12, y=1.02)
plt.show()

#### 4.1.1.2 Log models

In [ ]:
LPPDP_reldiff_lnmodels

In [ ]:
# Load Relative Difference from LPPDP (Performance of Pooled Models)
fig, ax = plt.subplots(figsize=(12, 6), dpi=150, constrained_layout=True)

# Compute differences from baseline model E
diffs = {
        'EA': (LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelEA']['rel_diff'].reset_index(drop=True) -
               LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'modelEAlnS': (LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelEAlnS']['rel_diff'].reset_index(drop=True) -
                LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'modelEAlnSZ': (LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelEAlnSZ']['rel_diff'].reset_index(drop=True) -
                 LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'modelEAlnST': (LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelEAlnST']['rel_diff'].reset_index(drop=True) -
                 LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'modelEAlnSP': (LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelEAlnSP']['rel_diff'].reset_index(drop=True) -
                 LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'modelEAlnSTWI': (LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelEAlnSTWI']['rel_diff'].reset_index(drop=True) -
                 LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'modelEAlnTWIGEO': (LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelEAlnTWIGEO']['rel_diff'].reset_index(drop=True) -
                 LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'modelEAlnSZT': (LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelEAlnSZT']['rel_diff'].reset_index(drop=True) -
                  LPPDP_reldiff_lnmodels[LPPDP_reldiff_lnmodels.model == 'modelE']['rel_diff'].reset_index(drop=True)),
}

colors = {
    'EA':      'royalblue',     # strong blue
    'modelEAlnS':     'crimson',       # vivid red
    'modelEAlnSZ':    'seagreen',      # green with blue tint
    'modelEAlnST':    'darkorange',    # deep orange
    'modelEAlnSP':    'mediumvioletred', # magenta-pink
    'modelEAlnSTWI':  'slateblue',     # purple-blue
    'modelEAlnTWIGEO':'black',         # neutral black
    'modelEAlnSZT':   'saddlebrown',   # earthy brown
}

markers = {
    'EA': 's',       # square
    'modelEAlnS': 'o',      # circle
    'modelEAlnSZ': '^',     # triangle up
    'modelEAlnST': 'D',     # diamond
    'modelEAlnSP': 'v',     # triangle down
    'modelEAlnSTWI': '>',   # triangle right
    'modelEAlnTWIGEO': 'X', # x-shaped
    'modelEAlnSZT': '<',    # triangle left
}



# Sort models by final relative difference
sorted_models = sorted(diffs.keys(), key=lambda m: diffs[m].iloc[-1])

# Plot stacked differences
previous_diff = pd.Series([0] * len(pSA_T[:12]))  # Baseline

for model in sorted_models:
        current_diff = diffs[model]
        ax.plot(pSA_T[:12], current_diff, marker=markers[model], linestyle='-', color=colors[model],
                markersize=6, alpha=0.9, label=f'Model {model}')
        ax.fill_between(pSA_T[:12], previous_diff, current_diff, color=colors[model], alpha=0.4)
        previous_diff = current_diff

# Baseline line and annotation
ax.axhline(y=0, xmin=0.001, xmax=6, linestyle='-', color='black', linewidth=3, zorder=0)
ax.text(0.12, 0.2, 'Baseline: LPPD of isotropic model E',
            fontsize=10,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', linewidth=0.8))

# Axis configuration
ax.set_ylabel('Relative LPPD Difference (%)')
ax.set_xlabel('Period T [s]')
ax.set_xscale('log')
ax.set_xlim([0.01, 6])
ax.set_ylim([0,5])
ax.set_ylim(bottom=0)
ax.grid(which='both', linestyle='-', linewidth=0.75, color='gray', alpha=0.2)

# Explicit ticks
ax.set_xticks([0.01, 0.03, 0.06, 0.1, 0.3, 1.0, 3.0, 6.0])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.get_xaxis().set_minor_formatter(plt.NullFormatter())

# Legend
ax.legend(loc='upper left', fontsize=10, frameon=True, framealpha=0.8)

# Title
# fig.suptitle(r'Relative Difference in LPPD', fontsize=12, y=1.02)
plt.show()

### 4.1.2 Selected Models

In [ ]:
# Load Relative Difference from LPPDP (Performance of Pooled Models)
fig, ax = plt.subplots(figsize=(12, 6), dpi=150, constrained_layout=True)

# Compute differences from baseline model E
diffs = {
        'EA': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEA']['rel_diff'].reset_index(drop=True) -
               LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EAS': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEAS']['rel_diff'].reset_index(drop=True) -
                LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EASZ': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEASZ']['rel_diff'].reset_index(drop=True) -
                 LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EAST': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEAST']['rel_diff'].reset_index(drop=True) -
                 LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
        'EASZT': (LPPDP_reldiff[LPPDP_reldiff.model == 'modelEASZT']['rel_diff'].reset_index(drop=True) -
                  LPPDP_reldiff[LPPDP_reldiff.model == 'modelE']['rel_diff'].reset_index(drop=True)),
}

# Define colors
colors = {
        'EA':      'royalblue',      # strong blue
        'EAS':     'crimson',        # vivid red
        'EASZ':    'seagreen',       # green with blue tint
        'EAST':    'darkorange',     # deep orange
        'EASZT':   'saddlebrown',    # earthy brown
}

# Define distinct markers
markers = {
        'EA': 's',      # square
        'EAS': 'o',     # circle
        'EASZ': '^',    # triangle up
        'EAST': 'D',   # diamond
        'EASZT': '<',  # triangle left
}

# Sort models by final relative difference
sorted_models = sorted(diffs.keys(), key=lambda m: diffs[m].iloc[-1])

# Plot stacked differences
previous_diff = pd.Series([0] * len(pSA_T[:12]))  # Baseline

for model in sorted_models:
        current_diff = diffs[model]
        ax.plot(pSA_T[:12], current_diff, marker=markers[model], linestyle='-', color=colors[model],
                markersize=6, alpha=0.9, label=f'Model {model}')
        ax.fill_between(pSA_T[:12], previous_diff, current_diff, color=colors[model], alpha=0.4)
        previous_diff = current_diff

# Baseline line and annotation
ax.axhline(y=0, xmin=0.001, xmax=6, linestyle='-', color='black', linewidth=3, zorder=0)
ax.text(0.12, 0.2, 'Baseline: LPPD of isotropic model E',
            fontsize=10,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', linewidth=0.8))

# Axis configuration
ax.set_ylabel('Relative LPPD Difference (%)')
ax.set_xlabel('Period T [s]')
ax.set_xscale('log')
ax.set_xlim([0.01, 6])
ax.set_ylim([0,5])
ax.set_ylim(bottom=0)
ax.grid(which='both', linestyle='-', linewidth=0.75, color='gray', alpha=0.2)

# Explicit ticks
ax.set_xticks([0.01, 0.03, 0.06, 0.1, 0.3, 1.0, 3.0, 6.0])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.get_xaxis().set_minor_formatter(plt.NullFormatter())

# Legend
ax.legend(loc='upper left', fontsize=10, frameon=True, framealpha=0.8)

# Title
# fig.suptitle(r'Relative Difference in LPPD', fontsize=12, y=1.02)
plt.show()

## 4.2 Event-Specific Model Performance

### 4.2.1 Append Additional event information: tectonic class and magnitude

In [ ]:
gm_unique = gm_im_table_50[['evid','mag','tect_class']].drop_duplicates('evid')
LPPDES = LPPDES.merge(gm_unique, on='evid', how='left')

### 4.2.2 All Models

In [ ]:
# Define periods
psa_EES = ['PGA','pSA_0.03','pSA_0.06','pSA_0.1','pSA_0.3','pSA_0.6','pSA_1.0','pSA_2.0','pSA_3.0','pSA_4.0','pSA_5.0','pSA_6.0']
psa_EEES = ['E_PGA','E_pSA_0.03','E_pSA_0.06','E_pSA_0.1','E_pSA_0.3','E_pSA_0.6','E_pSA_1.0','E_pSA_2.0','E_pSA_3.0','E_pSA_4.0','E_pSA_5.0','E_pSA_6.0']
psa_EAES = ['EA_PGA','EA_pSA_0.03','EA_pSA_0.06','EA_pSA_0.1','EA_pSA_0.3','EA_pSA_0.6','EA_pSA_1.0','EA_pSA_2.0','EA_pSA_3.0','EA_pSA_4.0','EA_pSA_5.0','EA_pSA_6.0']
psa_EASES = ['EAS_PGA','EAS_pSA_0.03','EAS_pSA_0.06','EAS_pSA_0.1','EAS_pSA_0.3','EAS_pSA_0.6','EAS_pSA_1.0','EAS_pSA_2.0','EAS_pSA_3.0','EAS_pSA_4.0','EAS_pSA_5.0','EAS_pSA_6.0']
psa_EASZES = ['EASZ_PGA','EASZ_pSA_0.03','EASZ_pSA_0.06','EASZ_pSA_0.1','EASZ_pSA_0.3','EASZ_pSA_0.6','EASZ_pSA_1.0','EASZ_pSA_2.0','EASZ_pSA_3.0','EASZ_pSA_4.0','EASZ_pSA_5.0','EASZ_pSA_6.0']
psa_EASTES = ['EAST_PGA','EAST_pSA_0.03','EAST_pSA_0.06','EAST_pSA_0.1','EAST_pSA_0.3','EAST_pSA_0.6','EAST_pSA_1.0','EAST_pSA_2.0','EAST_pSA_3.0','EAST_pSA_4.0','EAST_pSA_5.0','EAST_pSA_6.0']
psa_EASPES = ['EASP_PGA','EASP_pSA_0.03','EASP_pSA_0.06','EASP_pSA_0.1','EASP_pSA_0.3','EASP_pSA_0.6','EASP_pSA_1.0','EASP_pSA_2.0','EASP_pSA_3.0','EASP_pSA_4.0','EASP_pSA_5.0','EASP_pSA_6.0']
psa_EASTWIES = ['EASTWI_PGA','EASTWI_pSA_0.03','EASTWI_pSA_0.06','EASTWI_pSA_0.1','EASTWI_pSA_0.3','EASTWI_pSA_0.6','EASTWI_pSA_1.0','EASTWI_pSA_2.0','EASTWI_pSA_3.0','EASTWI_pSA_4.0','EASTWI_pSA_5.0','EASTWI_pSA_6.0']
psa_EATWIGEOES = ['EATWIGEO_PGA','EATWIGEO_pSA_0.03','EATWIGEO_pSA_0.06','EATWIGEO_pSA_0.1','EATWIGEO_pSA_0.3','EATWIGEO_pSA_0.6','EATWIGEO_pSA_1.0','EATWIGEO_pSA_2.0','EATWIGEO_pSA_3.0','EATWIGEO_pSA_4.0','EATWIGEO_pSA_5.0','EATWIGEO_pSA_6.0']
psa_EASZTES = ['EASZT_PGA','EASZT_pSA_0.03','EASZT_pSA_0.06','EASZT_pSA_0.1','EASZT_pSA_0.3','EASZT_pSA_0.6','EASZT_pSA_1.0','EASZT_pSA_2.0','EASZT_pSA_3.0','EASZT_pSA_4.0','EASZT_pSA_5.0','EASZT_pSA_6.0']

vmin, vmax = 4.5, 7.5

# helper scatter function
def scatter_panel(ax, x, y, c, marker, label=None):
    return ax.scatter(
        x, y,
        c=c,
        cmap='coolwarm',
        vmin=vmin, vmax=vmax,
        marker=marker,
        edgecolor='k',
        lw=0.1,
        label=label
    )

def format_ticks_as_int(val, pos):
    return f"{int(val):d}" if val >= 1 else ""

# assign markers for tectonic classes
tect_markers = {
    'Crustal': 'o',
    'Interface': 's',
    'Slab': '^'
}

pSA_plot_idx = 11
fig, ax = plt.subplots(3,3, figsize=(12,10), dpi=200, constrained_layout=True)

# loop over tectonic classes
scat = None
for tect, marker in tect_markers.items():
    mask = LPPDES.tect_class == tect
    # top-left panel as example
    scat = scatter_panel(
        ax[0,0],
        LPPDES.n_recs[mask],
        ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EEES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100,
        LPPDES.mag[mask],
        marker,
        label=tect
    )
    # repeat for other subplots
    scatter_panel(ax[0,1], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EAES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)
    scatter_panel(ax[0,2], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EASES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker, label = tect)
    scatter_panel(ax[1,0], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EASZES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)
    scatter_panel(ax[1,1], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EASTES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)
    scatter_panel(ax[1,2], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EASPES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)
    scatter_panel(ax[2,0], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EASTWIES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)
    scatter_panel(ax[2,1], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EATWIGEOES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)
    scatter_panel(ax[2,2], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EASZTES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)

# formatting
for i in range(3):
    ax[i,0].set_ylabel('Relative Difference in LPPD (%)')
    for j in range(3):
        ax[i,j].set_xscale('log')
        ax[i,j].grid(True, which='both', linestyle='--', alpha=0.5)
        ax[i,j].set_ylim([-30, 30])
        ax[i,j].xaxis.set_major_formatter(FuncFormatter(format_ticks_as_int))
        ax[i,j].tick_params(axis='x', which='major')
        ax[i,j].set_xticks([40, 60, 80, 100, 200])
        ax[i,j].plot([LPPDES.n_recs.min(), LPPDES.n_recs.max()], [0, 0], '--k')
        ax[i,j].set_xlim(LPPDES.n_recs.min(), LPPDES.n_recs.max())

# titles
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASP')
ax[2,0].set_title('Model EASTWI')
ax[2,1].set_title('Model EATWIGEO')
ax[2,2].set_title('Model EASZT')
ax[2,1].set_xlabel('Number of Records')

# shared colorbar
cbar = fig.colorbar(scat, ax=fig.axes, orientation='vertical', fraction=0.05, pad=0.02, aspect=40)
cbar.set_label('Magnitude')

# legend for tectonic class in top-right panel
ax[0,2].legend(title='Tectonic Class', loc='best')

plt.show()

### 4.2.3 Selected Models

In [ ]:
pSA_plot_idx = 11
fig, ax = plt.subplots(2,3, figsize=(12,6), dpi=200, constrained_layout=True)

# loop over tectonic classes
scat = None
for tect, marker in tect_markers.items():
    mask = LPPDES.tect_class == tect
    # top-left panel as example
    scat = scatter_panel(
        ax[0,0],
        LPPDES.n_recs[mask],
        ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EEES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100,
        LPPDES.mag[mask],
        marker,
        label=tect
    )
    # repeat for other subplots
    scatter_panel(ax[0,1], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EAES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)
    scatter_panel(ax[0,2], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EASES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker, label = tect)
    scatter_panel(ax[1,0], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EASZES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)
    scatter_panel(ax[1,1], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EASTES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)
    scatter_panel(ax[1,2], LPPDES.n_recs[mask], ((LPPDES[psa_EES[pSA_plot_idx]][mask] - PO_LPPDE[psa_EASZTES[pSA_plot_idx]][mask]) / LPPDES[psa_EES[pSA_plot_idx]][mask]) * 100, LPPDES.mag[mask], marker)

# formatting
for i in range(2):
    ax[i,0].set_ylabel('Relative Difference in LPPD (%)')
    for j in range(3):
        ax[i,j].set_xscale('log')
        ax[i,j].grid(True, which='both', linestyle='--', alpha=0.5)
        ax[i,j].set_ylim([-20, 20])
        ax[i,j].xaxis.set_major_formatter(FuncFormatter(format_ticks_as_int))
        ax[i,j].tick_params(axis='x', which='major')
        ax[i,j].set_xticks([40, 60, 80, 100, 200])
        ax[i,j].plot([LPPDES.n_recs.min(), LPPDES.n_recs.max()], [0, 0], '--k')
        ax[i,j].set_xlim(LPPDES.n_recs.min(), LPPDES.n_recs.max())

# titles
ax[0,0].set_title('Model E')
ax[0,1].set_title('Model EA')
ax[0,2].set_title('Model EAS')
ax[1,0].set_title('Model EASZ')
ax[1,1].set_title('Model EAST')
ax[1,2].set_title('Model EASZT')
ax[1,0].set_xlabel('Number of Records')
ax[1,1].set_xlabel('Number of Records')
ax[1,2].set_xlabel('Number of Records')

# shared colorbar
cbar = fig.colorbar(scat, ax=fig.axes, orientation='vertical', fraction=0.05, pad=0.02, aspect=40)
cbar.set_label('Magnitude')

# legend for tectonic class in top-right panel
ax[0,0].legend(title='Tectonic Class', loc='best')

plt.show()